# 🔧 Project 3.1 — Sensor Data Cleaner
**DSA for Mechatronics · Week 03**

> **Your task:** Run every cell from top to bottom.  
> At the end, a full report is printed automatically.  
> **Submit:** A screenshot or PDF of the final report cell output.  
> **Present:** Be ready to explain what each step does and why.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ║   This seeds your personal dataset — be honest!     ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

# ── seed everything from your ID ──────────────────────
import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready — run the next cells.")


## Step 1 — Generate your personal sensor signal

In [ ]:
# Motor temperature sensor — 300 readings (one per second)
# Normal behaviour: ~22 °C with small Gaussian noise
# Fault behaviour : random 5-10% of readings spike to 60–160 °C

N            = 300
NORMAL_TEMP  = round(random.uniform(18.0, 28.0), 1)   # your motor's idle temp
SPIKE_PROB   = round(random.uniform(0.04, 0.12), 3)   # 4–12% spike rate
SPIKE_LOW    = round(random.uniform(55.0, 80.0),  1)
SPIKE_HIGH   = round(random.uniform(120.0, 160.0), 1)

raw_signal = [
    round(NORMAL_TEMP + random.gauss(0, 0.5), 2)
    if random.random() > SPIKE_PROB
    else round(random.uniform(SPIKE_LOW, SPIKE_HIGH), 2)
    for _ in range(N)
]

print(f"Signal parameters:")
print(f"  Normal temperature : {NORMAL_TEMP} °C")
print(f"  Spike probability  : {SPIKE_PROB*100:.1f}%")
print(f"  Spike range        : {SPIKE_LOW} – {SPIKE_HIGH} °C")
print(f"  Total readings     : {N}")
print(f"  Actual spikes (>50°C): {sum(1 for x in raw_signal if x > 50)}")
print()
print("First 20 readings:")
for i, v in enumerate(raw_signal[:20]):
    flag = " ← SPIKE" if v > 50 else ""
    print(f"  t={i:03d}: {v:7.2f} °C{flag}")


## Step 2 — Detect spikes using a sliding window

In [ ]:
def detect_spikes(signal, window_size=5, threshold=15.0):
    """
    A reading is a spike if it differs from its neighbours' average by more than threshold.
    Neighbours: up to window_size readings on each side, excluding the reading itself.
    """
    spike_indices = []
    for i in range(len(signal)):
        neighbours = signal[max(0, i - window_size) : i] + signal[i + 1 : i + 1 + window_size]
        if neighbours:
            avg = sum(neighbours) / len(neighbours)
            if abs(signal[i] - avg) > threshold:
                spike_indices.append(i)
    return spike_indices

spikes = detect_spikes(raw_signal)

print(f"Spike detection (sliding window, size=5, threshold=15°C):")
print(f"  Spikes found : {len(spikes)}")
print(f"  Spike rate   : {len(spikes)/N*100:.1f}%")
print()
print(f"  {'Index':>6}  {'Raw value':>10}  {'Type'}")
print(f"  {'─'*6}  {'─'*10}  {'─'*20}")
for idx in spikes[:12]:
    print(f"  {idx:6d}  {raw_signal[idx]:>9.2f}°C  SPIKE")
if len(spikes) > 12:
    print(f"  ... and {len(spikes)-12} more")


## Step 3 — Clean the signal (interpolation)

In [ ]:
def clean_signal(signal, spike_indices):
    """
    Replace each spike with the average of its nearest non-spike neighbours.
    """
    spike_set = set(spike_indices)
    cleaned   = list(signal)
    for i in spike_indices:
        l = i - 1
        while l >= 0 and l in spike_set:
            l -= 1
        r = i + 1
        while r < len(signal) and r in spike_set:
            r += 1
        lv = signal[l] if l >= 0           else signal[i]
        rv = signal[r] if r < len(signal)  else signal[i]
        cleaned[i] = round((lv + rv) / 2, 2)
    return cleaned

cleaned = clean_signal(raw_signal, spikes)

print("Signal cleaning results:")
print(f"  Raw    — min: {min(raw_signal):7.2f}°C   max: {max(raw_signal):7.2f}°C")
print(f"  Cleaned— min: {min(cleaned):7.2f}°C   max: {max(cleaned):7.2f}°C")
print()
print("Before / after at spike locations:")
print(f"  {'Index':>6}  {'Before':>9}  {'After':>9}")
print(f"  {'─'*6}  {'─'*9}  {'─'*9}")
for idx in spikes[:8]:
    print(f"  {idx:6d}  {raw_signal[idx]:>8.2f}°C  {cleaned[idx]:>8.2f}°C")


## Step 4 — Find the longest stable operating segment (two pointers)

In [ ]:
def longest_stable_segment(signal, tolerance=2.0):
    """
    Find the longest contiguous window where every reading stays
    within ±tolerance of the first value in that window.
    Returns (start, end, length).
    """
    best_start, best_len, left = 0, 1, 0
    for right in range(len(signal)):
        if abs(signal[right] - signal[left]) > tolerance:
            left = right
        if right - left + 1 > best_len:
            best_start = left
            best_len   = right - left + 1
    return best_start, best_start + best_len - 1, best_len

seg_start, seg_end, seg_len = longest_stable_segment(cleaned, tolerance=2.0)
seg_values = cleaned[seg_start : seg_end + 1]

print(f"Longest stable segment (±2.0°C tolerance):")
print(f"  Start    : t = {seg_start} s")
print(f"  End      : t = {seg_end} s")
print(f"  Duration : {seg_len} s")
print(f"  Mean     : {sum(seg_values)/len(seg_values):.2f} °C")
print(f"  Spread   : {min(seg_values):.2f} – {max(seg_values):.2f} °C")


## 📋 Final Report — this is what you submit

In [ ]:
# ── compute a few more stats for the report ──────────────────────────
avg_raw     = sum(raw_signal) / len(raw_signal)
avg_cleaned = sum(cleaned) / len(cleaned)
status      = "✅ NOMINAL" if max(cleaned) < 60 else "⚠️  WARNING"

W = 52
print("╔" + "═"*W + "╗")
print("║" + " MOTOR TEMPERATURE SENSOR — ANALYSIS REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<22}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<22}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  SIGNAL PARAMETERS" + " "*(W-19) + "║")
print(f"║  {'Normal temperature':<22}: {NORMAL_TEMP:<5.1f} °C{'':<19}║")
print(f"║  {'Spike probability':<22}: {SPIKE_PROB*100:<5.1f} %{'':<19}║")
print(f"║  {'Spike range':<22}: {SPIKE_LOW:.1f} – {SPIKE_HIGH:.1f} °C{'':<12}║")
print(f"║  {'Total readings':<22}: {N:<26}║")
print("╠" + "═"*W + "╣")
print("║  CLEANING RESULTS" + " "*(W-18) + "║")
print(f"║  {'Spikes detected':<22}: {len(spikes):<4} ({len(spikes)/N*100:.1f}%){'':<14}║")
print(f"║  {'Raw max temp':<22}: {max(raw_signal):<7.2f} °C{'':<16}║")
print(f"║  {'Cleaned max temp':<22}: {max(cleaned):<7.2f} °C{'':<16}║")
print(f"║  {'Average (raw)':<22}: {avg_raw:<7.2f} °C{'':<16}║")
print(f"║  {'Average (cleaned)':<22}: {avg_cleaned:<7.2f} °C{'':<16}║")
print("╠" + "═"*W + "╣")
print("║  STABLE SEGMENT" + " "*(W-16) + "║")
print(f"║  {'Start':<22}: t = {seg_start:<3} s{'':<20}║")
print(f"║  {'End':<22}: t = {seg_end:<3} s{'':<20}║")
print(f"║  {'Duration':<22}: {seg_len:<4} s{'':<20}║")
print("╠" + "═"*W + "╣")
print(f"║  STATUS : {status:<{W-11}}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Answer the three questions below **in this cell** (double-click to edit).  
There is no single correct answer — write in your own words, 2–4 sentences each.

---

**Q1 — What does your data show?**  
Describe the key numbers from your report: spike rate, fault locations, bearing period, longest run — whatever is relevant for your project. What does it tell you about the machine or system being monitored?

*Your answer here:*

---

**Q2 — Which algorithm or data structure did you find most interesting, and why?**  
Pick one step from the notebook (sliding window, prefix sum, two pointers, frequency map, pattern matching…) and explain in your own words how it works and what problem it solves.

*Your answer here:*

---

**Q3 — What would break if the student ID changed?**  
If a classmate used your notebook but changed the student ID to their own, which specific numbers in the final report would be different, and why?

*Your answer here:*
